# 3d-point-cloud (unet branch) -- Colab training

Trains the sparse 3D U-Net backbone experiment (`configs/exp_sparse_unet.yaml`, experiment 5 in the dense/sparse/SlotFormer-depth comparison -- see `README.md`'s "Backbone registry" section), and optionally the other 4 experiments via `run_all_experiments.py`.

**Before running for real**: `exp_sparse_unet.yaml`'s `BATCH_SIZE: 8` is an *unmeasured* starting guess (see its own comment) -- run the "quick batch-size safety check" section below first and adjust it if needed. Every other `exp_*.yaml` was tuned this same way (see their comments for the numbers measured on an RTX 2070) -- Colab GPUs (T4/A100/etc) are different hardware, so don't assume those numbers transfer.

Runtime -> Change runtime type -> GPU, before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone and install

The model code lives on the `unet` branch specifically -- a plain `git clone` without `-b unet` checks out `main` instead and won't have `backbone3d_unet.py`/`exp_sparse_unet.yaml`.

In [ ]:
!git clone -b unet https://github.com/izione/3d-point-cloud.git
%cd 3d-point-cloud
!pip install -q -r requirements.txt

Optional: spconv accelerates the sparse backbones if it installs cleanly for this Colab image's CUDA version (`nvcc --version` or `!nvidia-smi` shows it) -- everything works without it too (pure-PyTorch fallback, `models/backbone3d_auto.py` probes automatically), so skip this cell if the install fails or you're not sure which `cuXXX` tag matches.

In [ ]:
# !pip install -q spconv-cu126   # pick the cuXXX tag matching this runtime's CUDA (see https://github.com/traveller59/spconv)

## 2. Mount the dataset

Upload the dataset (`PersonX/scene_XXXX/{sonar,labels}`, matching `data/dataset.py`'s `SonarDiverDataset` layout) to Google Drive first, then set `DRIVE_DATASET_PATH` below to wherever it landed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATASET_PATH = "/content/drive/MyDrive/dataset"  # <-- edit to wherever you uploaded PersonX/scene_XXXX/

In [ ]:
# Point DATA.ROOT at the Drive path instead of hand-editing the yaml.
import yaml

with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["DATA"]["ROOT"] = DRIVE_DATASET_PATH
with open("configs/default.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("DATA.ROOT ->", DRIVE_DATASET_PATH)

## 3. Sanity check (synthetic data, no dataset needed)

Confirms every `configs/exp_*.yaml` -- including `exp_sparse_unet.yaml` -- constructs, runs forward/backward, and produces finite losses, before touching the real dataset.

In [ ]:
!python smoke_test.py

## 4. Quick batch-size safety check (real data, sparse_unet)

Runs a handful of real steps and reports peak GPU memory -- same check every other `exp_*.yaml`'s `BATCH_SIZE` comment cites a number from. `exp_sparse_unet.yaml`'s U-Net backbone (encoder+decoder, roughly 2x the conv layers of the encoder-only backbones at a similar width) has never been measured on real hardware, so don't skip this before committing to a full run.

If `peak reserved` is above ~80% of this runtime's total GPU memory, lower `BATCH_SIZE` in `configs/exp_sparse_unet.yaml` (and scale `LR` down proportionally -- see the config's own comment for the linear-scaling convention this project uses) and re-run this cell.

In [ ]:
import torch
from torch.utils.data import DataLoader

from config_utils import load_config
from data.dataset import SonarDiverDataset, collate_fn
from models.detector import DiverDetector

BATCH_SIZE_TO_TEST = 8   # matches exp_sparse_unet.yaml's current guess -- change here to try a different value
N_STEPS = 40

device = torch.device("cuda")
cfg = load_config("configs/exp_sparse_unet.yaml")
ds = SonarDiverDataset(cfg, "train")
loader = DataLoader(ds, batch_size=BATCH_SIZE_TO_TEST, shuffle=True, collate_fn=collate_fn, drop_last=True)
model = DiverDetector(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

torch.cuda.reset_peak_memory_stats()
it = iter(loader)
for i in range(N_STEPS):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)
    losses, pred, stem_coords, assign_result = model.loss(batch, device)
    optimizer.zero_grad()
    losses["total"].backward()
    optimizer.step()
    if (i + 1) % 10 == 0:
        reserved = torch.cuda.max_memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"step {i+1}/{N_STEPS}: peak_reserved={reserved:.2f} GiB / {total:.1f} GiB ({100*reserved/total:.0f}%)  stem_voxels={stem_coords.shape[0]}")

del model, optimizer, loader, ds
torch.cuda.empty_cache()

## 5. Train

Checkpoints/logs go straight to Drive (`--ckpt_dir`) so a Colab disconnect mid-epoch doesn't lose progress -- `CKPT_EVERY_N_EPOCHS`/`CKPT_EVERY_N_STEPS` in the config control how often that happens.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/3d-point-cloud-checkpoints/sparse_unet"

!python train.py --config configs/exp_sparse_unet.yaml --ckpt_dir "{CKPT_DIR}" --exp_name sparse_unet

Resume after a disconnect (picks up the schedule/step count from the checkpoint -- see `train.py`'s own `--resume` handling):

In [ ]:
# !python train.py --config configs/exp_sparse_unet.yaml --ckpt_dir "{CKPT_DIR}" --exp_name sparse_unet --resume "{CKPT_DIR}/sparse_unet_last.pth"

## 6. Optional: run all 5 experiments back-to-back

`run_all_experiments.py` runs dense / sparse-only / sparse+SlotFormer(3L) / sparse+SlotFormer(6L) / sparse U-Net in sequence, each into its own subdirectory under `--ckpt_dir`'s parent via its own `<name>` (see the script's docstring). Point `--ckpt_dir` names at Drive the same way as above if you use this instead of the single `train.py` call in step 5.

In [ ]:
# !python run_all_experiments.py --only sparse_unet
# !python run_all_experiments.py   # all 5 -- see BATCH_SIZE caveats above for each config first

## 7. Evaluate

```bash
!python test.py --checkpoint "{CKPT_DIR}/sparse_unet_last.pth" --split test
```

See `README.md`'s "Test / evaluate" section for `--pr_curve_out` (PR-curve-per-IoU-threshold plot) and `eval_pr_comparison.py` for comparing multiple checkpoints (e.g. this U-Net run vs. `exp_sparse_slotformer_3l`) on one figure.